# SPoRC Two-Stage Summarization Pipeline (thousands of episodes trial)

This notebook contains:
1. Dataset construction + pseudo-label generation
2. Extractor + Refiner model (Two-Stage)
3. Training loop (trial on first 250 episodes)
4. Evaluation on pseudo summaries

All logic is embedded in a single notebook for debugging & iteration.

In [ ]:
import json
import random
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import nltk

from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import AutoModel, BartForConditionalGeneration
from collections import defaultdict
from rouge_score import rouge_scorer
from transformers import (
    AutoTokenizer,
    AutoModel,
    BartForConditionalGeneration
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cpu


In [ ]:
# ===== cell 1: dataset_tds.py (embedded in notebook) =====

class SPoRCDataset:
    """
    Dataset layer for SPoRC.

    Responsibilities:
    1. Load turn-level CSV and group by episode
    2. Load episode-level summaries from JSONL
    3. Provide role-aware pseudo-labels for extractor training
    4. Enforce training rules:
       - global_summary MUST exist, otherwise skip episode
       - host / guest trained only if corresponding summary exists
    """

    def __init__(
        self,
        csv_path: str,
        summary_jsonl_path: str,
        bart_name: str = "facebook/bart-large",
        sent_emb_name: str = "all-mpnet-base-v2",
        device: str = "cpu",
    ):
        # ---- load CSV ----
        self.df = pd.read_csv(csv_path)
        self.episodes = self._group_by_episode(self.df)

        # ---- tokenizer (used later by models) ----
        self.tokenizer = AutoTokenizer.from_pretrained(bart_name)

        # ---- sentence embedder (for pseudo-labels) ----
        self.sent_embedder = SentenceTransformer(sent_emb_name)
        self.sent_embedder = self.sent_embedder.to(device)

        # ---- load summaries ----
        self.summaries = {}
        self._load_summaries_jsonl(summary_jsonl_path)

    # ------------------------------------------------------------------
    # internal helpers
    # ------------------------------------------------------------------

    def _group_by_episode(self, df: pd.DataFrame):
        episodes = defaultdict(list)

        for _, row in df.iterrows():
            ep = row["episode"]
            episodes[ep].append(
                {
                    "turn": int(row["turn"]),
                    "role": row["role"],
                    "speaker": row["speaker"],
                    "text": row["text"],
                }
            )

        for ep in episodes:
            episodes[ep] = sorted(episodes[ep], key=lambda x: x["turn"])

        return episodes

    def _load_summaries_jsonl(self, path: str):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                ep = obj["episode"]

                self.summaries[ep] = {
                    "global": obj.get("global_summary", "").strip(),
                    "host": obj.get("host_summary", "").strip(),
                    "guest": obj.get("guest_summary", "").strip(),
                }

    # ------------------------------------------------------------------
    # public API
    # ------------------------------------------------------------------

    def episode_ids(self):
        return list(self.episodes.keys())

    def get_summary(self, episode: str, summary_type: str):
        if episode not in self.summaries:
            return ""
        return self.summaries[episode].get(summary_type, "")

    def encode_episode(self, episode: str, max_utt_len: int = 64):
        utts = self.episodes[episode]
        texts = [u["text"] for u in utts]
        roles = [u["role"] for u in utts]

        enc = self.tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_utt_len,
            return_tensors="pt",
        )

        return enc["input_ids"], enc["attention_mask"], roles, texts

    def build_pseudo_labels(
        self,
        episode: str,
        summary_type: str = "global",
        topk: int = 5,
    ):
        # ---- episode must have summaries ----
        if episode not in self.summaries:
            print(f"[WARN] episode {episode}: summary not found, skip training")
            return None

        summaries = self.summaries[episode]

        # ---- global summary is mandatory ----
        if summaries["global"] == "":
            print(
                f"[WARN] episode {episode}: global_summary missing, skip training"
            )
            return None

        # ---- role-specific summary ----
        summary_text = summaries.get(summary_type, "").strip()
        if summary_text == "":
            return None

        # ---- select utterances ----
        utts = self.episodes[episode]

        if summary_type == "host":
            idx_text = [(i, u["text"]) for i, u in enumerate(utts) if u["role"] == "host"]
        elif summary_type == "guest":
            idx_text = [(i, u["text"]) for i, u in enumerate(utts) if u["role"] == "guest"]
        else:
            idx_text = [(i, u["text"]) for i, u in enumerate(utts)]

        if len(idx_text) == 0:
            return None

        indices, texts = zip(*idx_text)

        # ---- sentence similarity ----
        utt_emb = self.sent_embedder.encode(list(texts), convert_to_tensor=True)
        sum_emb = self.sent_embedder.encode(summary_text, convert_to_tensor=True)

        sims = util.cos_sim(utt_emb, sum_emb).squeeze(1)

        k = min(topk, sims.size(0))
        topk_local = torch.topk(sims, k=k).indices.tolist()

        topk_global = [indices[i] for i in topk_local]
        return topk_global


In [ ]:
# =====cell 2: model_tds.py (embedded in notebook) =====

class UtteranceEncoder(nn.Module):
    """
    Encode each utterance into a fixed-size vector
    using a pretrained BART encoder.
    """
    def __init__(self, bart_name="facebook/bart-large"):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(bart_name).encoder
        self.encoder.to(device)
        
    def forward(self, input_ids, attention_mask):
        """
        input_ids:      [num_utts, seq_len]
        attention_mask: [num_utts, seq_len]

        returns:
            utt_emb: [num_utts, hidden_dim]
        """
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        # 使用每句话的第一个 token（<s>）作为句向量
        utt_emb = outputs.last_hidden_state[:, 0]
        return utt_emb


class Extractor(nn.Module):
    """
    Simple extractor:
    Given utterance embeddings, predict an importance score
    for each utterance.
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, utt_emb):
        """
        utt_emb: [num_utts, hidden_dim]

        returns:
            logits: [num_utts]
        """
        logits = self.classifier(utt_emb).squeeze(-1)
        return logits


class Refiner(nn.Module):
    """
    Abstractive summarizer based on BART.
    """
    def __init__(self, bart_name="facebook/bart-large"):
        super().__init__()
        self.model = BartForConditionalGeneration.from_pretrained(bart_name)

    def forward(self, input_ids, attention_mask, labels=None):
        """
        Standard BART forward.

        If labels is provided, returns training loss.
        """
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            return_dict=True
        )


class TwoStageSummarizer(nn.Module):
    """
    Two-stage summarization model:
    1) UtteranceEncoder + Extractor (sentence selection)
    2) Refiner (BART generation)
    """
    def __init__(self, bart_name="facebook/bart-large"):
        super().__init__()

        self.encoder = UtteranceEncoder(bart_name)
        self.extractor = Extractor(hidden_dim=1024)  # bart-large hidden size
        self.refiner = Refiner(bart_name)

    def forward_extractor(self, input_ids, attention_mask):
        """
        Forward pass for extractor only.

        returns:
            logits:  [num_utts]
            utt_emb: [num_utts, hidden_dim]
        """
        utt_emb = self.encoder(input_ids, attention_mask)
        logits = self.extractor(utt_emb)
        return logits, utt_emb


In [ ]:
# =====cell 3 =====
def train_extractor(
    model,
    dataset,
    episodes,
    summary_type="global",
    topk=5,
    lr=1e-4,
):
    model.encoder.eval()      # encoder frozen
    model.extractor.train()

    optimizer = torch.optim.AdamW(model.extractor.parameters(), lr=lr)
    criterion = torch.nn.BCEWithLogitsLoss()

    total_loss = 0.0
    count = 0

    for ep in episodes:
        pos_indices = dataset.build_pseudo_labels(
            ep, summary_type=summary_type, topk=topk
        )
        if pos_indices is None:
            continue

        input_ids, attn, _, _ = dataset.encode_episode(ep)
        input_ids = input_ids.to(device)
        attn = attn.to(device)

        with torch.no_grad():
            utt_emb = model.encoder(input_ids, attn)

        logits = model.extractor(utt_emb)
        labels = torch.zeros_like(logits)
        labels[pos_indices] = 1.0

        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        count += 1

    return total_loss / max(count, 1)


In [ ]:
# =====cell 4: inference_tds.py (embedded in notebook) =====
SUMMARY_PREFIX = {
    "global": "<GLOBAL>",
    "host": "<HOST>",
    "guest": "<GUEST>",
}

def generate_summary(
    model,
    dataset,
    episode,
    summary_type="global",
    max_sentences=6,
    max_len=256,
):
    model.eval()

    input_ids, attn, _, texts = dataset.encode_episode(episode)
    input_ids = input_ids.to(device)
    attn = attn.to(device)

    with torch.no_grad():
        logits, _ = model.forward_extractor(input_ids, attn)
        topk = torch.topk(
            logits, k=min(max_sentences, logits.size(0))
        ).indices

    selected = [texts[i] for i in sorted(topk.tolist())]

    prefix = SUMMARY_PREFIX[summary_type]
    concat_text = prefix + " " + " ".join(selected)

    tok = dataset.tokenizer
    enc = tok(
        concat_text,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        out = model.refiner.model.generate(
            **enc,
            max_length=150,
            num_beams=4,
        )

    return tok.decode(out[0], skip_special_tokens=True)


In [ ]:
# =====cell 5: evaluation.py (embedded in notebook) =====

scorer = rouge_scorer.RougeScorer(
    ["rougeL"], use_stemmer=True
)


def eval_episode(
    model,
    dataset,
    episode,
    summary_type="global",
):
    """
    Evaluate ONE episode using ROUGE-L.
    """
    ref = dataset.get_summary(episode, summary_type)
    if ref == "":
        return None

    pred = generate_summary(
        model, dataset, episode, summary_type=summary_type
    )

    score = scorer.score(ref, pred)
    return score["rougeL"].fmeasure


def evaluate(
    model,
    dataset,
    episodes,
    summary_type="global",
):
    """
    Evaluate multiple episodes.
    """
    scores = []
    for ep in episodes:
        s = eval_episode(model, dataset, ep, summary_type)
        if s is not None:
            scores.append(s)

    if len(scores) == 0:
        return None

    return sum(scores) / len(scores)


In [ ]:
# ===== cell 6 =====
SUMMARY_PREFIX = {
    "global": "<GLOBAL>",
    "host": "<HOST>",
    "guest": "<GUEST>",
}

def train_refiner_on_episode(
    model,
    dataset,
    episode,
    summary_type,
    optimizer,
    max_len=256,
):
    ref = dataset.get_summary(episode, summary_type)
    if ref is None or ref.strip() == "":
        return None

    input_ids, attn, _, texts = dataset.encode_episode(episode)
    input_ids = input_ids.to(device)
    attn = attn.to(device)

    with torch.no_grad():
        logits, _ = model.forward_extractor(input_ids, attn)
        topk = torch.topk(logits, k=min(6, logits.size(0))).indices

    selected = [texts[i] for i in sorted(topk.tolist())]
    prefix = SUMMARY_PREFIX[summary_type]
    concat_text = prefix + " " + " ".join(selected)

    tok = dataset.tokenizer
    enc = tok(
        concat_text,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    ).to(device)

    tgt = tok(
        ref,
        truncation=True,
        max_length=150,
        return_tensors="pt",
    ).to(device)

    model.refiner.train()
    out = model.refiner(
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"],
        labels=tgt["input_ids"],
    )

    loss = out.loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()


In [ ]:
# ===== Cell 7: Main Training + Generation =====

# ROUGE-L Evaluation
# Initialize ROUGE scorer
rouge_scorer_l = rouge_scorer.RougeScorer(
    ["rougeL"], use_stemmer=True
)

def compute_rouge_l(pred, ref):
    """
    Compute ROUGE-L F1 score between a generated summary and a reference summary.
    The reference summary is treated as a silver (LLM-generated) gold.
    """
    if ref is None or ref.strip() == "":
        return None
    score = rouge_scorer_l.score(ref, pred)
    return score["rougeL"].fmeasure

# -------- 0. Setup --------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------------------------
# 1. Build Dataset
# -------------------------------------------------
dataset = SPoRCDataset(
    csv_path="sporc_turns_selected_clean.csv",
    summary_jsonl_path="sporc_full_summaries_7b.jsonl",
    device=device,
)

# -------------------------------------------------
# 2. Build Model
# -------------------------------------------------
model = TwoStageSummarizer().to(device)

# -------------------------------------------------
# 3. Add control tokens for Refiner (run once)
# -------------------------------------------------
SUMMARY_PREFIX = {
    "global": "<GLOBAL>",
    "host": "<HOST>",
    "guest": "<GUEST>",
}

dataset.tokenizer.add_tokens(["<GLOBAL>", "<HOST>", "<GUEST>"])
model.refiner.resize_token_embeddings(len(dataset.tokenizer))

# -------------------------------------------------
# 4. Train / Test split
# -------------------------------------------------
episodes_with_summary = list(dataset.summaries.keys())

train_episodes = episodes_with_summary[:250]
test_episodes  = episodes_with_summary[250:300]

print(f"Train episodes: {len(train_episodes)}")
print(f"Test episodes : {len(test_episodes)}")

# =================================================
# Part A: Train Extractor
# =================================================
print("\n===== Train Extractor =====")

for st in ["global", "host", "guest"]:
    loss = train_extractor(
        model,
        dataset,
        train_episodes,
        summary_type=st,
    )
    print(f"[Extractor] {st} average loss: {loss:.4f}")



# =================================================
# Part C: Train Refiner
# =================================================
print("\n===== Train Refiner =====")

refiner_optimizer = torch.optim.AdamW(
    model.refiner.parameters(), lr=2e-5
)

for st in ["global", "host", "guest"]:
    losses = []
    for ep in train_episodes:
        loss = train_refiner_on_episode(
            model,
            dataset,
            ep,
            summary_type=st,
            optimizer=refiner_optimizer,
        )
        if loss is not None:
            losses.append(loss)

    if len(losses) > 0:
        print(f"[Refiner] {st} average loss: {sum(losses)/len(losses):.4f}")
    else:
        print(f"[Refiner] {st}: no valid samples")

# =================================================
# Part D: Test Refiner with ROUGE-L
# =================================================
print("\n===== Evaluate Refiner (ROUGE-L) =====")

rouge_scores = {
    "global": [],
    "host": [],
    "guest": [],
}

for ep in test_episodes:
    for st in ["global", "host", "guest"]:
        # Generate summary
        pred = generate_summary(
            model, dataset, ep, summary_type=st
        )

        # Silver reference summary (LLM-generated)
        gold = dataset.get_summary(ep, st)

        score = compute_rouge_l(pred, gold)
        if score is not None:
            rouge_scores[st].append(score)

# Report average ROUGE-L
for st in ["global", "host", "guest"]:
    if len(rouge_scores[st]) > 0:
        avg = sum(rouge_scores[st]) / len(rouge_scores[st])
        print(f"ROUGE-L ({st}): {avg:.4f}")
    else:
        print(f"ROUGE-L ({st}): N/A")

# =================================================
# Part E: Generate One Example (for QA / Conflict)
# =================================================
ep = test_episodes[0]

global_summary = generate_summary(
    model, dataset, ep, summary_type="global"
)
host_summary = generate_summary(
    model, dataset, ep, summary_type="host"
)
guest_summary = generate_summary(
    model, dataset, ep, summary_type="guest"
)

print("\n--- Example Summaries ---")
print("\n[GLOBAL]\n", global_summary)
print("\n[HOST]\n", host_summary)
print("\n[GUEST]\n", guest_summary)

In [ ]:
# ========cell 8
# ===== load MNLI model =====
nli_model_name = "microsoft/deberta-v3-large-mnli"
nli_tokenizer = AutoTokenizer.from_pretrained(nli_model_name)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_model_name).to(device)
nli_model.eval()

LABEL_MAP = {
    0: "CONTRADICTION",
    1: "ENTAILMENT",
    2: "NEUTRAL",
}

@torch.no_grad()
def nli_predict(premise, hypothesis):
    inputs = nli_tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    logits = nli_model(**inputs).logits
    probs = F.softmax(logits, dim=-1)[0]
    label_id = torch.argmax(probs).item()

    return {
        "label": LABEL_MAP[label_id],
        "score": probs[label_id].item(),
        "probs": {
            LABEL_MAP[i]: probs[i].item() for i in range(3)
        }
    }


In [ ]:
# =========== Cell 9: Local Conflict Evidence (FIXED) =====
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")

sent_encoder = SentenceTransformer("all-MiniLM-L6-v2").to(device)

def split_sentences(text):
    return nltk.sent_tokenize(text)

@torch.no_grad()
def find_conflict_evidence(host_summary, guest_summary, topk=3):
    host_sents = split_sentences(host_summary)
    guest_sents = split_sentences(guest_summary)

    if len(host_sents) == 0 or len(guest_sents) == 0:
        return []

    host_emb = sent_encoder.encode(
        host_sents, convert_to_tensor=True
    )
    guest_emb = sent_encoder.encode(
        guest_sents, convert_to_tensor=True
    )

    sim = util.cos_sim(host_emb, guest_emb)

    pairs = []
    max_pairs = min(
        topk,
        sim.size(0) * sim.size(1)
    )

    # top-k pair 
    sim_copy = sim.clone()
    for _ in range(max_pairs):
        idx = torch.argmax(sim_copy)
        h, g = divmod(idx.item(), sim_copy.size(1))
        pairs.append((host_sents[h], guest_sents[g]))
        sim_copy[h, g] = -1

    evidence = []
    for h_sent, g_sent in pairs:
        nli_res = nli_predict(h_sent, g_sent)
        if nli_res["label"] == "CONTRADICTION":
            evidence.append({
                "host_sentence": h_sent,
                "guest_sentence": g_sent,
                "confidence": nli_res["score"],
            })

    return evidence


In [ ]:
# =========== Cell 10: Final Structured Output =====

def build_conflict_output(episode, host_summary, guest_summary):
    overall = nli_predict(host_summary, guest_summary)
    evidence = find_conflict_evidence(
        host_summary, guest_summary
    )

    return {
        "episode": episode,
        "conflict_overall": {
            "label": overall["label"],
            "score": overall["score"],
        },
        "conflict_type": (
            "stance_disagreement"
            if overall["label"] == "CONTRADICTION"
            else "none"
        ),
        "evidence": evidence,
    }


In [ ]:
# =============== Cell 11: Q&A =================

def answer_question(question, episode_result):
    q = question.lower()

    # ---- unpack with safety ----
    summaries = episode_result.get("summaries", {})
    conflict  = episode_result.get("conflict", {})

    conflict_label = conflict.get("label", "NEUTRAL")
    evidence = conflict.get("evidence", [])

    # ---- Conflict details (must be BEFORE general conflict) ----
    if "what" in q and "disagree" in q:
        if conflict_label != "CONTRADICTION":
            return "No specific disagreement was identified in this episode."

        if len(evidence) == 0:
            return (
                "A disagreement was detected, but no specific "
                "evidence sentence pair was identified."
            )

        e = evidence[0]
        return (
            "The disagreement is reflected in the following statements:\n"
            f"- Host: {e.get('host_sentence', '')}\n"
            f"- Guest: {e.get('guest_sentence', '')}"
        )

    # ---- Conflict existence ----
    if "disagree" in q or "conflict" in q:
        if conflict_label == "CONTRADICTION":
            return (
                "Yes. A disagreement between the host and the guest "
                "was detected."
            )
        else:
            return (
                "No clear disagreement between the host and the guest "
                "was detected."
            )

    # ---- Role-specific viewpoints ----
    if "host" in q and ("view" in q or "opinion" in q):
        return summaries.get("host", "No host-specific summary available.")

    if "guest" in q and ("view" in q or "opinion" in q):
        return summaries.get("guest", "No guest-specific summary available.")

    # ---- Global summary ----
    if "summary" in q or "about" in q:
        return summaries.get("global", "No global summary available.")

    # ---- Fallback ----
    return "This question is not supported by the current QA system."
